In [5]:
import glob
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"font.size": 12, "figure.autolayout": True})

# 1. Load and combine all summary files
results_dir = "./full-probe-sweep-q16"
summary_files = glob.glob(os.path.join(results_dir, "probe_*/summary.csv"))

if not summary_files:
    print(f"No summary.csv files found under {results_dir}. Check folder path.")
    exit(1)

all_data = []
for f in summary_files:
    strat_name = os.path.basename(os.path.dirname(f)).replace("probe_", "")
    df = pd.read_csv(f)
    df["probe_strategy"] = strat_name
    all_data.append(df)

master_df = pd.concat(all_data, ignore_index=True)

# Convert scores to percentage (0.0 - 1.0 -> 0 - 100%)
if master_df["score"].max() <= 1.0:
    master_df["score_pct"] = master_df["score"] * 100
else:
    master_df["score_pct"] = master_df["score"]

# 2. Filter / Deduplicate SDPA
santapp_df = master_df[master_df["backend"] == "santapp"].copy()
sdpa_df = master_df[master_df["backend"] == "sdpa"].copy()

# Average SDPA scores across probe runs per task (since they are identical dense baselines)
sdpa_consolidated = sdpa_df.groupby("task")["score_pct"].mean().reset_index()
sdpa_consolidated["backend"] = "SDPA Baseline"
sdpa_consolidated["probe_strategy"] = "SDPA Baseline"

# Reorder probe strategies logically for SANTA++
probe_order = ["start", "middle", "end", "random"]
santapp_df["probe_strategy"] = pd.Categorical(
    santapp_df["probe_strategy"], categories=probe_order, ordered=True
)

# ---------------------------------------------------------
# Plot 1: Overall Average Accuracy Comparison
# ---------------------------------------------------------
plt.figure(figsize=(9, 5.5))

# Calculate average accuracy per category
santa_avg = santapp_df.groupby("probe_strategy", observed=False)["score_pct"].mean().reset_index()
santa_avg["Category"] = santa_avg["probe_strategy"].apply(lambda x: f"SANTA++ ({x})")

sdpa_overall_avg = sdpa_consolidated["score_pct"].mean()
baseline_row = pd.DataFrame([{"Category": "SDPA (Baseline)", "score_pct": sdpa_overall_avg}])

bar_df = pd.concat([baseline_row, santa_avg[["Category", "score_pct"]]], ignore_index=True)

ax = sns.barplot(
    data=bar_df,
    x="Category",
    y="score_pct",
    palette=["#4C72B0"] + ["#DD8452"] * 4,
    edgecolor="black",
    alpha=0.9
)

plt.title("SANTA++ Probe Robustness (16 queries) vs. Single SDPA Baseline", fontsize=14, fontweight="bold", pad=12)
plt.xlabel("Evaluation Configuration", fontsize=12, fontweight="bold")
plt.ylabel("Mean Accuracy (%)", fontsize=12, fontweight="bold")
plt.ylim(0, 105)

# Add value labels on top of bars
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f"{height:.1f}%",
                    (p.get_x() + p.get_width() / 2., height),
                    ha="center", va="bottom", fontsize=10, xytext=(0, 3),
                    textcoords="offset points")

plt.xticks(rotation=15, ha="right")
plt.savefig("Benchmark Images/overall_accuracy_single_sdpa_q16.png", dpi=300)
print("Saved: overall_accuracy_single_sdpa_q16.png")
plt.close()

# ---------------------------------------------------------
# Plot 2: Per-Task Heatmap (Single SDPA Column)
# ---------------------------------------------------------
plt.figure(figsize=(10, 8))

# Pivot SANTA++: Tasks x Probe Strategy (fixed line below)
santa_pivot = santapp_df.pivot(index="task", columns="probe_strategy", values="score_pct")
santa_pivot.columns = [f"SANTA++ ({col})" for col in santa_pivot.columns]

# Prepare single SDPA column
sdpa_col = sdpa_consolidated.set_index("task")[["score_pct"]]
sdpa_col.columns = ["SDPA Baseline"]

# Combine into a single matrix with SDPA as the first column
heatmap_df = pd.concat([sdpa_col, santa_pivot], axis=1)

sns.heatmap(
    heatmap_df,
    annot=True,
    fmt=".1f",
    cmap="YlGnBu",
    cbar_kws={"label": "Accuracy (%)"},
    linewidths=0.5
)

plt.title("RULER Subtask Accuracy: SDPA Baseline vs. SANTA++ Probes (16 Queries)", fontsize=14, fontweight="bold", pad=12)
plt.xlabel("Configuration", fontsize=11, fontweight="bold")
plt.ylabel("Subtask", fontsize=11, fontweight="bold")
plt.savefig("Benchmark Images/task_heatmap_single_sdpa_q16.png", dpi=300)
print("Saved: task_heatmap_single_sdpa_q16.png")
plt.close()

C:\Users\theep\AppData\Local\Temp\ipykernel_31632\4074957745.py:63: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(


Saved: overall_accuracy_single_sdpa_q16.png
Saved: task_heatmap_single_sdpa_q16.png
